In [4]:
from pathlib import Path

base_local_path = Path(
    r"C:\Users\nshui\python\Kaggle\stores sales"
)


In [4]:
# 基本ライブラリ
import numpy as np
import pandas as pd

# 可視化
import matplotlib.pyplot as plt
import seaborn as sns

# モデル
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor

# 日付処理
from datetime import datetime


In [7]:
from pathlib import Path
import pandas as pd

# CSVファイルが入っているフォルダーを指定（例）
base_local_path = Path(r"C:\Users\nshui\python\Kaggle\stores sales")

# 指定先にCSVがあるか確認
print("フォルダーは存在するか:", base_local_path.exists())
print("CSVファイル:", [p.name for p in base_local_path.glob("*.csv")])

フォルダーは存在するか: True
CSVファイル: ['holidays_events.csv', 'oil.csv', 'sample_submission.csv', 'stores.csv', 'test.csv', 'train.csv', 'transactions.csv']


In [8]:
# 各 CSV を読み込む
train = pd.read_csv(base_local_path / "train.csv", parse_dates=["date"])
test = pd.read_csv(base_local_path / "test.csv", parse_dates=["date"])
stores = pd.read_csv(base_local_path / "stores.csv")
oil = pd.read_csv(base_local_path / "oil.csv", parse_dates=["date"])
holidays = pd.read_csv(
    base_local_path / "holidays_events.csv",
    parse_dates=["date"]
)
print("✔ データ読み込み完了")


✔ データ読み込み完了


In [11]:
# 原油価格の欠損を前方埋めで補完
oil["dcoilwtico"] = oil["dcoilwtico"].ffill()

print("✔ 原油価格の欠損補完完了")


✔ 原油価格の欠損補完完了


In [12]:
# transferred が True の祝日は除外し、Transfer の日を祝日扱いにする
holidays_clean = holidays.copy()

# transferred=True の行を除外
holidays_clean = holidays_clean[holidays_clean['transferred'] == False]

# type=Transfer の行は祝日扱いに変更
holidays_clean.loc[holidays_clean['type'] == 'Transfer', 'type'] = 'Holiday'

# 必要な列だけ残す
holidays_clean = holidays_clean[['date', 'type']].drop_duplicates()

print("✔ 祝日データの整理完了")


✔ 祝日データの整理完了


In [13]:
# store_nbr をキーに店舗メタデータを結合
train = train.merge(stores, on='store_nbr', how='left')
test = test.merge(stores, on='store_nbr', how='left')

print("✔ 店舗メタデータ結合完了")


✔ 店舗メタデータ結合完了


In [14]:
# 日付で原油価格を結合
train = train.merge(oil, on='date', how='left')
test = test.merge(oil, on='date', how='left')

print("✔ 原油価格結合完了")


✔ 原油価格結合完了


In [15]:
# 祝日フラグを作成
# 重複する列名（type）を避けるため、事前に列名を変更して結合します
holidays_temp = holidays_clean.rename(columns={'type': 'holiday_type'})
train = train.merge(holidays_temp, on='date', how='left')
test = test.merge(holidays_temp, on='date', how='left')

# holiday_type の有無を 0/1 に変換
train['is_holiday'] = train['holiday_type'].notnull().astype(int)
test['is_holiday'] = test['holiday_type'].notnull().astype(int)

# holiday_type は不要なので削除
train = train.drop(columns=['holiday_type'])
test = test.drop(columns=['holiday_type'])

print("✔ 祝日フラグ結合完了")

✔ 祝日フラグ結合完了


In [16]:
# 日付から特徴量を作成
for df in [train, test]:
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['dayofweek'] = df['date'].dt.dayofweek
    df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)

print("✔ 日付特徴量作成完了")


✔ 日付特徴量作成完了


C:\Users\nshui\AppData\Local\Temp\ipykernel_26720\636594741.py:6: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df['dayofweek'] = df['date'].dt.dayofweek
C:\Users\nshui\AppData\Local\Temp\ipykernel_26720\636594741.py:6: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df['dayofweek'] = df['date'].dt.dayofweek


In [17]:
# family, city, state, type をカテゴリ → 数値へ
# メモリ上で一度 'type' がドロップされてしまっている場合を考慮し、存在する列のみ処理します
cat_cols = ['family', 'city', 'state', 'type']

for col in cat_cols:
    if col in train.columns:
        train[col] = train[col].astype('category').cat.codes
    if col in test.columns:
        test[col] = test[col].astype('category').cat.codes

print("✔ カテゴリ変数のエンコード完了")

✔ カテゴリ変数のエンコード完了


In [18]:
# 目的変数
y = train['sales']

# 説明変数（sales と date は除外）
X = train.drop(columns=['sales', 'date'])

# test 用の説明変数
X_test = test.drop(columns=['date'])

print("✔ 学習データセット準備完了")


✔ 学習データセット準備完了


In [19]:
# 学習データを train / validation に分割
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("✔ train/validation 分割完了")


✔ train/validation 分割完了


In [20]:
import gc

# 1. データのコピーと型変換によるメモリ軽量化
train_clean = train.copy()
test_clean = test.copy()

# 2. カテゴリ変数の数値変換
y_clean = train_clean['sales'].astype('float32')
X_clean = train_clean.drop(columns=['sales', 'date'], errors='ignore')
X_test_clean = test_clean.drop(columns=['date'], errors='ignore')

for col in X_clean.columns:
    if X_clean[col].dtype == 'object' or isinstance(X_clean[col].dtype, pd.CategoricalDtype):
        categories = pd.concat([X_clean[col], X_test_clean[col]], axis=0).astype('category').cat.categories
        X_clean[col] = pd.Categorical(X_clean[col], categories=categories).codes
        X_test_clean[col] = pd.Categorical(X_test_clean[col], categories=categories).codes

# 数値列のデータ型をダウングレードしてメモリを節約 (float64 -> float32, int64 -> int32)
for col in X_clean.columns:
    if X_clean[col].dtype == 'float64':
        X_clean[col] = X_clean[col].astype('float32')
        X_test_clean[col] = X_test_clean[col].astype('float32')
    elif X_clean[col].dtype == 'int64':
        X_clean[col] = X_clean[col].astype('int32')
        X_test_clean[col] = X_test_clean[col].astype('int32')

# 不要になった中間変数を削除してメモリ解放
del train_clean, test_clean
gc.collect()

# 3. Train/Validation 分割
X_train_clean, X_val_clean, y_train_clean, y_val_clean = train_test_split(
    X_clean, y_clean, test_size=0.2, random_state=42
)

# 4. 軽量化したランダムフォレストモデル（メモリ節約のためにパラメータを調整）
model = RandomForestRegressor(
    n_estimators=50,       # 木の数を減らしてメモリ消費を抑える
    max_depth=15,          # 木が無限に深く成長するのを防ぎメモリを抑制
    min_samples_leaf=4,    # 葉の最小サンプル数を増やしてモデルサイズを縮小
    random_state=42,
    n_jobs=-1
)

print("⌛ 学習開始（メモリ最適化版）...")
model.fit(X_train_clean, y_train_clean)

# 変数を最新のものに更新
X_train, X_val, y_train, y_val = X_train_clean, X_val_clean, y_train_clean, y_val_clean
X_test = X_test_clean

del X_train_clean, X_clean
gc.collect()

print("✔ モデル学習完了（メモリクラッシュ防止対策済み）")

⌛ 学習開始（メモリ最適化版）...
✔ モデル学習完了（メモリクラッシュ防止対策済み）


In [21]:
# 予測
y_pred = model.predict(X_val)

# RMSE（評価指標） 0日近いほうがいい、MSEのルート版　100-150がかなり良いモデル　平均してRMSEくらい売上外してる
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(f"✔ Validation RMSE: {rmse:.4f}")


✔ Validation RMSE: 264.8139


In [23]:
# test.csv の予測
test_pred = model.predict(X_test)

# sample_submission の形式に合わせる
submission = pd.DataFrame({
    'id': test['id'],
    'sales': test_pred
})

submission_path = base_local_path / "submission.csv"
submission.to_csv(submission_path, index=False)

print(f"✓ 提出用データ作成＆保存完了: {submission_path}")
submission.head()

✓ 提出用データ作成＆保存完了: C:\Users\nshui\python\Kaggle\stores sales\submission.csv


,id,sales
0,3000888,4.417963
1,3000889,0.000917
2,3000890,6.451682
3,3000891,2364.234793
4,3000892,0.193268
